# Importado librerias

In [ ]:
import pandas as pd
import plotly.express as px

# Obtencion de datos de base de datos

Vamos a comenzar obteniendo todos los datos que necesito, de la base de datos ChEMBL v37. A diferencia de la version anterior (ChEMBL34), esta vez la base no esta como MySQL en un servidor inhouse, sino como un dump de SQLite (`chembl_37.db`, descargado de `ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/releases/chembl_37/`). SQLite no soporta `INTO OUTFILE`, asi que el mismo `WHERE` se corrio con `sqlite3`/`pandas` (`pd.read_sql_query`, por chunks) en un script aparte: `extract_chembl37.py`. El query es equivalente al original:

```sql
    SELECT act.molregno,
        md.chembl_id AS comp_id,
        ass.tid as tid,
        ass.assay_id as assay_id,
        trgd.chembl_id AS target,
        act.standard_relation AS relation,
        act.standard_value AS bioactivity,
        act.standard_units AS units,
        act.standard_type AS type,
        act.potential_duplicate,
        act.pchembl_value,
        trgd.organism,
        trgd.target_type,
        trgd.pref_name,
        ass.assay_type,
        cmpstc.canonical_smiles AS smiles,
        cmpstc.standard_inchi_key AS inchi_key,
        cmpprop.mw_freebase, cmpprop.alogp, cmpprop.hba, cmpprop.hbd, cmpprop.psa,
        cmpprop.rtb, cmpprop.ro3_pass, cmpprop.num_ro5_violations, cmpprop.full_mwt,
        cmpprop.aromatic_rings, cmpprop.heavy_atoms, cmpprop.qed_weighted,
        cmpprop.full_molformula, cmpprop.np_likeness_score,
        cs.sequence AS sequence,
        cs.organism AS sequence_organism
    FROM activities AS act
    JOIN assays AS ass ON act.assay_id = ass.assay_id
    JOIN target_dictionary AS trgd ON ass.tid = trgd.tid
    JOIN compound_structures AS cmpstc ON act.molregno = cmpstc.molregno
    JOIN molecule_dictionary AS md ON act.molregno = md.molregno
    JOIN compound_properties AS cmpprop ON md.molregno = cmpprop.molregno
    JOIN target_components AS tc ON ass.tid = tc.tid
    JOIN component_sequences AS cs ON tc.component_id = cs.component_id
    WHERE trgd.target_type='SINGLE PROTEIN'
    AND ass.assay_type='B'
    AND cmpprop.heavy_atoms>3
    AND act.standard_value IS NOT NULL
    AND trgd.organism IS NOT NULL
    ;
```

Nota de esquema: en ChEMBL37, `compound_properties` (antes `cmpprop.*`) perdio 9 columnas que tenia en ChEMBL34 -- `cx_most_apka, cx_most_bpka, cx_logp, cx_logd, molecular_species, mw_monoisotopic, hba_lipinski, hbd_lipinski, num_lipinski_ro5_violations` -- todas dependientes de ChemAxon o de los conteos Lipinski "originales". No se usan en ningun filtro de este notebook, asi que se dropearon directamente en el query en vez de traerlas vacias.

La idea es traer todos los datos que necesito, sin ningun filtro. Todos los filtros los quiero hacer directo en pandas y bien visuales.
Basicamente, lo que necesito son todos los compuestos ensayados, los targets contra los que fueron ensayados, los tipos de ensayos, etc.

Con todos estos datos la idea va a ser obtener sets de datos de compuestos quimicos ensayados contra cada uno los targets, que van a ser proteinas. Seria un set de datos por proteina, de modo que luego voy a poder entrenar algoritmos de ML para poder clasificarlos en activos e inactivos.

Entonces ahora arranco por importar todos estos datos crudos a pandas

# Limpieza inicial

Toda la limpieza inicial va a estar basada en el curado de datos hecho por Rifaioglu et al. 2020. Si bien hay cambios importantes respecto a varios criterios, el objetivo del curado es relativamente el mismo, obtener ensayos de compuestos quimicos contra proteina simple, y clasificarlos como Activos e inactivos.

Se arranca haciendo las limpiezas sensillas y duras.  
Decidimos que sean ensayos de proteina simple y de binding, después:  
- No me interesan los que no dicen de que organismo vienen (suelen ser datos raros mal pasados)
- No me interesan los que no tengan bioactividad
- Saco las sales y atomos simples (que tienen menos de 3 atomos pesados)

El query ya hace esos filtros!

In [ ]:
df = pd.read_csv("../data/raw/data_chembl37_full.csv")
df

In [ ]:
df.to_pickle("../data/raw/data_chembl37_full.pkl")

# Filtros basicos

Otro filtro bastante rapido que hago ahora, para poder descartar ensayos que no podria curar sin tener que ponerme a leer individualmente la descripcion de cada ensayo. Son todos aquellos que usan unidades MUY dependiente del ensayo que se planteo, tipo %, que no tengo manera de saber % respecto a que habla.

In [ ]:
px.bar(df.value_counts("units").head(10))

Son como 700k. Si estoy buscando un target especifico, podria ir a buscar ensayos ahi...  
Es más engorroso porque habria que leer los papers, asi que para hacer un filtrado muy general para buscar targets inespecificamente, los descarto.

A esta altura del curado de targets, solamente estoy buscando obtener a gran escala todos los targets que pueda viablemente.

In [ ]:
print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

df = df[df["units"]!="%"]

print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))

# Exploracion de tipos de ensayos

Necesito ver como estandarizar los ensayos, y ver que tipos de ensayos valen la pena quedarme y seguir limpiando despues

In [ ]:
# voy a quedarme SOLO para evaluar los tipos los 300 targets con más compuestos. Que tiene más de 500 compuestos (poco para poder entrenar buen una CNN) .
top_300 = df["comp_id"].value_counts().sort_values(ascending=False).head(300).index
df_300 = df[df["comp_id"].isin(top_300)]


In [ ]:
cantidad_ensayos = df_300.value_counts("type")
cantidad_ensayos = cantidad_ensayos[cantidad_ensayos>300]
px.bar(x=cantidad_ensayos.index,y=cantidad_ensayos)

Quiero ver cuales tipos de ensayos valen la pena evaluar, del top 100 de mas compuestos

In [ ]:
def barplot_cuantity_comps(df, column, cuantity=50, log_y = True):
    assays_count = df.groupby(column)["comp_id"].count().sort_values(ascending=False)
    fig = px.bar(assays_count.head(cuantity).sort_values(ascending=False), title=f'{column} assay counts', log_y=log_y)
    fig.add_trace(px.bar(x=assays_count.head(cuantity).index, y=assays_count.head(cuantity).values).data[0])
    fig.update_layout(xaxis=dict(tickfont=dict(size=10)),showlegend=False)
    return fig

In [ ]:
barplot_cuantity_comps(df_300,["type"])

## Me quedo con los tipos de ensayos faciles
Por el momento voy a seleccionar rapidamente los que es facil definir la concentracion por el tipo de ensayo asi puedo empezar a ver la relacion entre positivos y negativos

In [ ]:
print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")
assays = ('IC50','EC50','AC50','Ki','Kd','Potency')
df = df[df.type.isin(assays)]

print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

In [ ]:
df_easy_types_300 = df[df["comp_id"].isin(top_300)]

Podemos ver que la gran mayoria de ensayos tienen unidades nM y ug/ML. Entonces vamos a ver que podemos salvar.

In [ ]:
barplot_cuantity_comps(df_easy_types_300,"units")

voy a rescatar los ensayos faciles que valgan la pena!

# Estandarizacion de datos

La idea de esta seccion, seria estandarizar todos los ensayos a nM, para poder aplicar criterios claros de que es activo y que inactivo.
Como vi antes, casi todos los ensayos son de nM. Casi que podria descartar los que no... PERO como ya tenia esta funcion hecha la corri igual... quiza si aparecen mas ensayos de unidades raras, pero que valen la pena, se podria retocar.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def convert_to_standard_unit(row):
    unit = row["units"]

    if unit == "nM":
        return row["bioactivity"]

    elif unit == 'µM' or unit == 'uM':
        bioactivity = row["bioactivity"]
        return bioactivity*1000

    elif unit == '10^2 uM':
        bioactivity = row["bioactivity"]
        return bioactivity*100000
    
    elif unit == "ug.mL-1":
        try:
            smiles = row["smiles"]
            mw = Descriptors.MolWt(Chem.MolFromSmiles(smiles))
            bioactivity = row["bioactivity"]
            return bioactivity*((mw)**-1)*1e6
        except:
            comp = row["comp_id"]
            print(f"Unable to convert to nM {comp}")
    
    else:
        return "no_std_unit"


In [ ]:
df.loc[:,"std_bioactivity_nM"] = df.apply(convert_to_standard_unit,axis=1)

In [ ]:
print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

df = df[df["std_bioactivity_nM"] != "no_std_unit"]
df.loc[:,"bioactivity"] = df.loc[:,"std_bioactivity_nM"]
df.loc[:,"units"] = "nM"
df = df.drop(columns="std_bioactivity_nM")

print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

# Desduplicacion de datos por medianas

Como no quiero tener más de una bioactividad por target-compuesto lo que hago es de todos los compuestos que tengo, quedarme con la mediana.
OJO, esto solo se podría hacer con ensayos que sean equivalentes en cuanto a que más/menos compuesto significa lo mismo y sobre todo estén en las mismas unidades...  
**Entonces esto solo se puede correr despues de asegurarme que todos los ensayos esten en la misma unidad y sean equivalentes**

In [ ]:
def compid_target_desduplication_median(df):
    
    group = ['comp_id', 'target', 'relation', 'type','organism','assay_type']
    median_bioactivity = df.groupby(group,dropna=False)['bioactivity'].median().reset_index()
    median_bioactivity = median_bioactivity.rename(columns={'bioactivity': 'median_bioactivity'})
    df = df.drop_duplicates(group)
    df = pd.merge(df, median_bioactivity, on=group, how='left')
    df['bioactivity'] = df['median_bioactivity']
    df = df.drop(columns=['median_bioactivity'])
    return df

print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

df = compid_target_desduplication_median(df)

print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

# Binarizacion de bioactividad y limpieza profunda

In [ ]:
def step_func_bioactivity(row):
    relation = row['relation']
    bioactivity = row['bioactivity']
    if relation in ('=','<') and bioactivity <= 10000:
        return 1
    elif relation in ('=','>') and bioactivity >= 20000:
        return 0
    else:
        return 'indeterminate'

print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")
df.loc[:,'binary_bioactivity'] = df.loc[:,["relation","bioactivity"]].apply(step_func_bioactivity,axis=1)
df = df[df.binary_bioactivity != 'indeterminate']
print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

## Desduplicado de ensayos repetidos de discordantes en bioactivdad
Vemos que todavia quedan ensayos duplicados, tenemos que ver como hacer para que queden unicos.


In [ ]:
df[df.duplicated(["comp_id","target"],keep=False)].sort_values(["comp_id","target"])

Entonces, voy a hacer dos cosas, para intentar desduplicarlos:
1. Si estos ensayos respecto a la bioactividad, alguno (de los duplicados) es de Kd o Ki, confio mucho más. Esto es porque los ensayos de Ki y Kd siempre dan el mismo resultado para la misma enzima. Es una propiedad INTRINSECA de la enzima. Los IC50 como es relativo, puede variar segun temperatura del ambiente, mini concentraciones distintas de los reactivos, si una mariposa aletea correctamente en la turndra siberiana... Entonces primero de todo, veo si hay ensayo Ki o Kd me quedo con ese. 
2. Si todos los ensayos dan la misma bioactividad binaria, entonces todos concuerdan en ese dato, que es con el que yo termino entrenando los modelos como label. Entonces, no me cambia cual sea, siempre y cuando no haya discordancias. Me quedo con cualquiera de los puntos, y a desduplique

(pd. si estaa funcion bastante rebuscada, la hizo gpt, y funciona premium)

In [ ]:
from collections import Counter
def compid_target_desduplication_binary_bioact(df):
    
    df.sort_values('type')

    groups = df.groupby(['comp_id', 'target'])

    counter = Counter(unmodified=0,saved=0,droped=0)

    # define a function to apply to each group
    def drop_duplicates(group,counter_obj):
        # get the unique values of binary_bioactivity in the group
        bioactivity_values = group['binary_bioactivity'].unique()

        # if there is only one unique value (no hay discordancias de bioactividad), keep the first row in the group (cualquiera)
        if len(bioactivity_values) == 1:
            counter_obj["unmodified"] += 1
            return group.head(1)

        # if there are multiple unique values, prioritize rows with Ki or Kd in the type column
        else:
            ki_or_kd_rows = group[group['type'].isin(['Ki', 'Kd'])]
            if len(ki_or_kd_rows) > 0:
                counter_obj["saved"] += 1
                return ki_or_kd_rows.head(1)
            else:
                counter_obj["droped"] += 1
                return pd.DataFrame()

    # apply the function to each group and concatenate the results
    df_clean = pd.concat([drop_duplicates(group,counter) for _, group in groups])

    # reset the index of the DataFrame
    df_clean = df_clean.reset_index(drop=True)

    print(f"Assays droped: {counter}")

    return df_clean


print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

df = compid_target_desduplication_binary_bioact(df)

print('dp',len(df))
print('targets',len(df.target.unique()))
print('compuestos',len(df.comp_id.unique()))
print("")

In [ ]:
df.to_pickle("../data/raw/data_chembl_37_filtrada_31_08_26.pkl")

In [ ]:
def barplot_cuantity_comps_binary_bioactivity(df,cuantity,log_y=True):
    bioact_count = df.groupby("target")["binary_bioactivity"].value_counts()
    best_X_count = df.groupby("target")["comp_id"].count().sort_values(ascending=False).head(cuantity)
    best_X_count.name = "comp_cuantity"
    bin_bioact_count_bestX = bioact_count[best_X_count.index]
    bin_bioact_count_bestX = bin_bioact_count_bestX.reset_index()
    bin_bioact_count_bestX.binary_bioactivity = bin_bioact_count_bestX.binary_bioactivity.astype("str")
    bin_bioact_count_bestX.dtypes
    bin_bioact_count_bestX = pd.merge(bin_bioact_count_bestX,df[["target","organism"]],how="inner",left_on="target",right_on="target").drop_duplicates()
    bin_bioact_count_bestX = pd.merge(bin_bioact_count_bestX,best_X_count,how="inner",left_on="target",right_on="target").drop_duplicates()
    fig_best_X_bbioact = px.bar(bin_bioact_count_bestX, x='target', y='count', hover_data=['organism','comp_cuantity',], color='binary_bioactivity', title='Binary Bioactivity Count per Target',
             labels={'count': 'Count', 'target_name':'Target','target': 'Id', 'binary_bioactivity': 'Binary Bioactivity', "organism": "Organism","comp_cuantity":"Total Assays"},
             category_orders={"binary_bioactivity": ["1", "0"]},
             color_discrete_map={"1": 'orange', "0": 'blue'},
             barmode='stack',
             log_y=log_y)

    return fig_best_X_bbioact

barplot_cuantity_comps_binary_bioactivity(df,100)